# V9 schema-3 observability

Run this notebook on a **high-RAM** runtime. RAM is the real constraint; a GPU runtime is not required and only costs compute units. The package has no CUDA path at all — the sole device reference is `device="cpu"` in `gnn/graphmodel_rgcn.py`, and this resume path does not train, it only scores from a verified checkpoint.

Drive holds the package and receives the final export; the active run uses local `/content` scratch.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

PACKAGE_CANDIDATES = [
    Path('/content/drive/MyDrive/v9_observability_colab_schema3'),
    Path('/content/drive/MyDrive/Colab Notebooks/v9_observability_colab_schema3'),
]
PACKAGE_ON_DRIVE = next(
    (str(path) for path in PACKAGE_CANDIDATES if path.is_dir()),
    str(PACKAGE_CANDIDATES[0]),
)
if not Path(PACKAGE_ON_DRIVE).is_dir():
    raise FileNotFoundError(
        'Could not find the uploaded package. Checked: '
        + ', '.join(str(path) for path in PACKAGE_CANDIDATES)
    )
LOCAL_PACKAGE = '/content/v9_observability_colab_schema3'
EXPORT_DIR = '/content/drive/MyDrive/v9_schema3_results'
print(f'Using package: {PACKAGE_ON_DRIVE}')

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

package_source = Path(PACKAGE_ON_DRIVE)
package_local = Path(LOCAL_PACKAGE)
if not package_source.is_dir():
    raise FileNotFoundError(f'Package folder is missing: {package_source}')
if package_local.is_symlink() or package_local.is_file():
    package_local.unlink()
elif package_local.exists():
    shutil.rmtree(package_local)
shutil.copytree(package_source, package_local)
requirements = package_local / 'requirements.txt'
if not requirements.is_file():
    raise FileNotFoundError(f'Package requirements are missing: {requirements}')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)],
    check=True,
)
print(f'Copied package to {package_local}')

The next cell probes the Drive export path **before** anything expensive runs. The producer only touches `EXPORT_DIR` in its last step, and `_export_bundle` publishes all-or-nothing: it stages the whole `recovery/` tree into a temp directory on the Drive mount, sha256-verifies every staged file by reading it back off Drive, and then swaps it in with `os.replace()` on a *directory* (moving any existing export aside first). If the mount cannot support those operations, the failure lands after hours of compute and the artifact is left on `/content` scratch, which is destroyed with the VM.

The probe replays that exact sequence — `copytree` with `copy2`, per-file read-back hashing, directory `os.replace()` over an existing non-empty target, and the separate `copy2` + hash-check + `replace()` used for the JSON — against a throwaway tree, then deletes everything it wrote. It costs a few seconds and raises on failure, so a broken or out-of-quota Drive mount stops "Run all" here.

In [ ]:
import hashlib
import os
import shutil
import tempfile
from pathlib import Path

# Fail in seconds, not after hours.
#
# run_schema3_observability.py::_export_bundle publishes the recovery/
# evidence tree by staging it inside EXPORT_DIR (a Drive FUSE mount),
# sha256-verifying every staged file by reading it back off Drive, and
# then swapping it into place with os.replace() on a *directory* -- with
# a move-aside step when the destination already exists. Those are the
# operations a FUSE mount is most likely to reject, and they only run at
# the very end of the producer, after all the compute. If they fail
# there, _export_bundle deletes its staging dir and re-raises, leaving
# the artifact on /content scratch only, where it dies with the VM.
#
# This probe runs the same sequence against a throwaway tree, so a Drive
# mount that cannot support the export surfaces now instead of then.


def _probe_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


export_dir = Path(EXPORT_DIR)
export_dir.mkdir(parents=True, exist_ok=True)

probe_source = Path('/content/.export_probe_src')
probe_target = export_dir / '.export_probe'
probe_aside = export_dir / f'.export_probe.old.{os.getpid()}.tmp'
probe_tmp_file = export_dir / f'.export_probe.{os.getpid()}.tmp'
probe_json = export_dir / '.export_probe.json'
stage_dir = None

if probe_source.exists():
    shutil.rmtree(probe_source)
(probe_source / 'bundles' / 'probe').mkdir(parents=True)
(probe_source / 'current.json').write_text('{"probe": true}', encoding='utf-8')
(probe_source / 'bundles' / 'probe' / 'manifest.json').write_text(
    '{"probe": true}', encoding='utf-8'
)
# A megabyte of incompressible bytes makes the read-back check meaningful:
# a truncated or silently buffered Drive write will not match.
(probe_source / 'bundles' / 'probe' / 'object.bin').write_bytes(os.urandom(1 << 20))

try:
    # Pre-create the destination so the move-aside branch is exercised too;
    # that is the path a rerun into an existing export dir actually takes.
    shutil.rmtree(probe_target, ignore_errors=True)
    probe_target.mkdir(parents=True)
    (probe_target / 'placeholder.txt').write_text('placeholder', encoding='utf-8')

    stage_dir = Path(tempfile.mkdtemp(dir=export_dir, prefix='.export-probe-stage-'))
    shutil.copytree(
        probe_source, stage_dir, copy_function=shutil.copy2, dirs_exist_ok=True
    )

    for source_file in sorted(p for p in probe_source.rglob('*') if p.is_file()):
        staged_file = stage_dir / source_file.relative_to(probe_source)
        if not staged_file.is_file():
            raise RuntimeError(f'staged file missing after copytree: {staged_file}')
        if _probe_sha256(staged_file) != _probe_sha256(source_file):
            raise RuntimeError(f'staged file failed read-back check: {staged_file}')

    os.replace(probe_target, probe_aside)
    os.replace(stage_dir, probe_target)
    stage_dir = None
    shutil.rmtree(probe_aside, ignore_errors=True)

    # _export_file() publishes the JSON separately: copy2 to a temp name in
    # the export dir, hash check, then Path.replace() onto the final name.
    shutil.copy2(probe_source / 'current.json', probe_tmp_file)
    if _probe_sha256(probe_source / 'current.json') != _probe_sha256(probe_tmp_file):
        raise RuntimeError('probe artifact hash does not match after copy to Drive')
    probe_tmp_file.replace(probe_json)
except Exception as exc:
    raise RuntimeError(
        'Drive export probe FAILED. run_schema3_observability.py::_export_bundle '
        f'performs these same operations against {export_dir} and would fail the '
        'same way at the very end of the run, after all the compute, leaving the '
        'artifact on VM-local scratch only. Fix the Drive mount (remount the '
        'drive, free up Drive quota, or choose a different EXPORT_DIR) before '
        f'continuing. Underlying error: {exc!r}'
    ) from exc
finally:
    if stage_dir is not None:
        shutil.rmtree(stage_dir, ignore_errors=True)
    shutil.rmtree(probe_aside, ignore_errors=True)
    shutil.rmtree(probe_target, ignore_errors=True)
    shutil.rmtree(probe_source, ignore_errors=True)
    probe_tmp_file.unlink(missing_ok=True)
    probe_json.unlink(missing_ok=True)

print(
    f'Drive export probe passed: {export_dir} supports copytree + copy2, '
    'sha256 read-back verification, and directory os.replace.'
)

The next cells set up a local Ollama server inside this Colab VM and hard-verify the exact model tag `gemma4:12b` before the producer runs. They install Ollama, start `ollama serve` as a detached background process, poll it until it accepts connections, pull `gemma4:12b`, and then re-parse `ollama list` using the *same* column-0/skip-header logic as the producer's `preflight_local_model` check (`gnn/explanation_narrative.py`) so this check cannot disagree with the producer. A cold 12B CPU load can take several minutes; the bounded CLI smoke probe allows 180 seconds and limits visible output. The probe checks CLI availability, flags, and cold-load generation only; it does not claim full selector-contract validation. Every step raises a Python exception on failure instead of printing a shell error and continuing, so "Run all" stops here rather than burning hours of compute on a run whose narrative preflight is doomed to fail.

Do not substitute another model. If `gemma4:12b` is a private/local tag that is not on the public Ollama registry, the pull step below will fail with instructions to import it manually (for example via a Modelfile, or by copying an existing `~/.ollama/models` blob store from Drive into this VM) before re-running these setup cells.

In [ ]:
import shutil
import subprocess


def _run_checked(command, label, *, timeout=600):
    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            f"{label} failed (exit {result.returncode})."
        )
    return result


# Recent Ollama installers use zstd to unpack the release. Stock Colab
# images do not always include it, so install the small system dependency
# before invoking the installer. Colab runs this notebook as root.
if shutil.which("zstd") is None:
    if shutil.which("apt-get") is None:
        raise RuntimeError("zstd is missing and apt-get is unavailable.")
    _run_checked(["apt-get", "update"], "apt-get update")
    _run_checked(
        ["apt-get", "install", "-y", "zstd"],
        "apt-get install zstd",
    )
else:
    print("zstd is already installed.")


def _ollama_cli_works():
    if shutil.which("ollama") is None:
        return False
    version = subprocess.run(
        ["ollama", "--version"],
        capture_output=True,
        text=True,
        timeout=30,
    )
    return version.returncode == 0


if not _ollama_cli_works():
    install = subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh",
        shell=True,
        capture_output=True,
        text=True,
        timeout=600,
    )
    print(install.stdout)
    print(install.stderr)
    if install.returncode != 0:
        raise RuntimeError(
            "Ollama install script failed (exit "
            f"{install.returncode}). See stderr above for details."
        )
else:
    print("Ollama CLI is already installed.")

if not _ollama_cli_works():
    raise RuntimeError("Ollama installation completed but the CLI is not usable.")
print("Ollama installed and usable.")

In [ ]:
import subprocess

# "ollama serve" is a long-running foreground process; a Colab code
# cell blocks until its process exits, so it must be launched detached
# with its output redirected to a log file instead of run with `!`.
OLLAMA_LOG_PATH = "/content/ollama_serve.log"
_ollama_log = open(OLLAMA_LOG_PATH, "w")
ollama_server_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=_ollama_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f"Started Ollama server, pid={ollama_server_process.pid}, log={OLLAMA_LOG_PATH}")

In [ ]:
import time
import urllib.error
import urllib.request

OLLAMA_BASE_URL = "http://127.0.0.1:11434"
READY_TIMEOUT_SECONDS = 60
POLL_INTERVAL_SECONDS = 1.0


def _ollama_server_is_up():
    try:
        with urllib.request.urlopen(OLLAMA_BASE_URL, timeout=3) as resp:
            return resp.status == 200
    except (urllib.error.URLError, ConnectionError, TimeoutError, OSError):
        return False


_deadline = time.monotonic() + READY_TIMEOUT_SECONDS
_ready = False
while time.monotonic() < _deadline:
    if _ollama_server_is_up():
        _ready = True
        break
    if ollama_server_process.poll() is not None:
        break
    time.sleep(POLL_INTERVAL_SECONDS)

if not _ready:
    try:
        with open(OLLAMA_LOG_PATH) as f:
            _log_tail = f.readlines()[-40:]
    except OSError:
        _log_tail = []
    print("".join(_log_tail))
    raise RuntimeError(
        f"Ollama server did not become ready at {OLLAMA_BASE_URL} "
        f"within {READY_TIMEOUT_SECONDS}s (log tail printed above, "
        f"full log at {OLLAMA_LOG_PATH})."
    )
print("Ollama server is ready.")

In [ ]:
import subprocess

# Must match gnn/explanation_narrative.py MODEL_TAG exactly.
MODEL_TAG = "gemma4:12b"

# If an imported/private exact-tag model is already present, do not ask
# the public registry for it. The later list check and generation smoke
# test still validate that the local model is usable.
listed_before_pull = subprocess.run(
    ["ollama", "list"],
    capture_output=True,
    text=True,
    timeout=10,
    check=False,
)
listed_lines = [
    line.split()
    for line in listed_before_pull.stdout.splitlines()
    if line.split()
]
installed_before_pull = {columns[0] for columns in listed_lines[1:]}
if listed_before_pull.returncode == 0 and MODEL_TAG in installed_before_pull:
    print(f"{MODEL_TAG} is already installed; skipping registry pull.")
else:
    pull = subprocess.run(
        ["ollama", "pull", MODEL_TAG],
        capture_output=True,
        text=True,
        timeout=1800,
    )
    print(pull.stdout)
    print(pull.stderr)
    if pull.returncode != 0:
        raise RuntimeError(
            f"`ollama pull {MODEL_TAG}` failed (exit {pull.returncode}). "
            f"{MODEL_TAG} may be a private/local tag that is not published "
            "to the public Ollama registry. Import the exact tag into this "
            "Colab VM's Ollama model store before rerunning this cell -- for "
            "example by copying an existing Ollama blob store from Drive. Do "
            "NOT substitute a different model tag: the producer's narrative "
            f"preflight check is pinned to `{MODEL_TAG}`, and a different tag "
            "will produce an artifact that fails validation."
        )
    print(f"{MODEL_TAG} pulled.")

In [ ]:
import subprocess

MODEL_TAG = "gemma4:12b"


def _installed_model_names(stdout):
    # Exact port of gnn/explanation_narrative.py::_installed_model_names --
    # split stdout into non-blank lines, split each line on whitespace,
    # take column 0, and skip the header row. This check must parse
    # `ollama list` identically to the producer's preflight_local_model,
    # or it can pass here and still fail inside the long run.
    lines_ = [line.split() for line in str(stdout).splitlines() if line.split()]
    return {columns[0] for columns in lines_[1:] if columns}


listed = subprocess.run(
    ["ollama", "list"],
    capture_output=True,
    text=True,
    timeout=10,
    check=False,
)
print(listed.stdout)
installed = _installed_model_names(listed.stdout)
if listed.returncode != 0 or MODEL_TAG not in installed:
    raise RuntimeError(
        f"local {MODEL_TAG} is unavailable "
        f"(ollama list exit={listed.returncode}, parsed models={sorted(installed)}). "
        "This mirrors gnn/explanation_narrative.py::preflight_local_model "
        "exactly, so the long run would record the preflight failure, use "
        "deterministic narrative fallbacks, and fail its coverage gate."
    )
print(f"Verified: {MODEL_TAG} is present in `ollama list` output.")

In [ ]:
import subprocess

MODEL_TAG = "gemma4:12b"

def _smoke_excerpt(value, limit=512):
    if value is None:
        return ""
    if isinstance(value, bytes):
        value = value.decode("utf-8", errors="replace")
    value = str(value).strip()
    return value if len(value) <= limit else value[:limit] + "...<truncated>"

_smoke_command = [
    "ollama",
    "run",
    MODEL_TAG,
    "--format",
    "json",
    "--think=false",
    "--keepalive",
    "10m",
    "--nowordwrap",
]
try:
    # This checks only CLI availability, flags, and cold-load generation.
    _smoke_result = subprocess.run(
        _smoke_command,
        input="Reply with exactly one word: ready",
        capture_output=True,
        text=True,
        timeout=180,
        check=False,
    )
except subprocess.TimeoutExpired as exc:
    raise RuntimeError(
        f"Ollama CLI smoke probe timed out after {exc.timeout}s. "
        f"stdout={_smoke_excerpt(getattr(exc, 'stdout', None) or getattr(exc, 'output', None))!r}; "
        f"stderr={_smoke_excerpt(getattr(exc, 'stderr', None))!r}"
    ) from exc
if _smoke_result.returncode != 0:
    raise RuntimeError(
        f"Ollama CLI smoke probe failed (return code {_smoke_result.returncode}); "
        f"stdout={_smoke_excerpt(_smoke_result.stdout)!r}; "
        f"stderr={_smoke_excerpt(_smoke_result.stderr)!r}"
    )
if not (_smoke_result.stdout or '').strip():
    raise RuntimeError("Ollama CLI smoke probe returned empty stdout; cold-load generation produced no output.")
print("Ollama CLI smoke probe passed (CLI/flags/cold-load only; selector-contract validation is outside this probe).")

This is the long cell. It runs the producer with `--allow-shortfall`.

Read what that flag does before trusting a green cell. Without it, `run_schema3_observability.py` treats a coverage-gate failure as fatal: it returns `1` and **skips the export**, so a degraded run leaves nothing on Drive. With it, the gate failure is still printed in full and still recorded in `work_root/result.json` as `coverage_gate_passed: false`, but the producer exits `0` and publishes the evidence anyway. Since `resume_observability()` has no resume path, rerunning this cell is a complete recompute — which is why the flag is set up front rather than kept in reserve.

The consequence: **a clean run of this cell no longer means the run was healthy.** The verification cell at the end reads `result.json` and reports the real verdict; check it. `--allow-shortfall` saves the evidence a degraded run did produce, it does not fill in explanations that were never generated.

A non-zero exit now means the producer actually crashed. If it crashed after writing the artifact but during the Drive publish, use the re-export cell below rather than rerunning this one.

In [ ]:
import subprocess
from pathlib import Path

RUN_ONE_CASE_DIAGNOSTIC = False
DIAGNOSTIC_WORK_ROOT = Path('/content/v9_schema3_diag3')
DIAGNOSTIC_LOG = Path('/content/v9_schema3_diag3.log')

if not RUN_ONE_CASE_DIAGNOSTIC:
    print(
        'Skipped one-case diagnostic (RUN_ONE_CASE_DIAGNOSTIC=False); Run All skips it. '
        'Set it to True and run this cell manually to execute the diagnostic.'
    )
    print(f'When enabled, combined live output will be saved to {DIAGNOSTIC_LOG}.')
else:
    diagnostic_command = [
        'python3',
        '-u',
        str(Path(LOCAL_PACKAGE) / 'run_schema3_observability.py'),
        '--package-root',
        LOCAL_PACKAGE,
        '--work-root',
        str(DIAGNOSTIC_WORK_ROOT),
        '--hybrid-detail-limit',
        '1',
        '--baseline-control-limit',
        '0',
        '--allow-shortfall',
    ]
    print('Running one-case diagnostic; live output is also saved to', DIAGNOSTIC_LOG)
    with DIAGNOSTIC_LOG.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(
            diagnostic_command,
            cwd=LOCAL_PACKAGE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f'One-case diagnostic exited with code {return_code}; '
            f'full combined output is in {DIAGNOSTIC_LOG}.'
        )
    print(f'One-case diagnostic completed successfully. Log: {DIAGNOSTIC_LOG}')


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

# The producer never clears its work root, and RecoveryBundleWriter stages
# evidence incrementally into it under a checkpoint.json. Reusing a dirty
# work root is therefore meaningful, not neutral:
#   * same run fingerprint  -> it RESUMES from the staged evidence
#   * different fingerprint -> RecoveryBundleError('recovery checkpoint
#     fingerprint mismatch') immediately
#   * an already-published bundle whose id matches but whose manifest does
#     not -> RecoveryBundleError('existing schema-3 bundle conflicts')
# None of those corrupt anything, but the last two abort the rerun, so the
# state is reported up front rather than diagnosed from a traceback. A fresh
# Colab VM always starts clean; this only matters when rerunning in place.
RESET_WORK_ROOT = False

WORK_ROOT = Path('/content/v9_schema3_run')
work_root = WORK_ROOT
staging_dir = work_root / '.hybrid_recovery_explanations_v9.recovery-stage'
recovery_dir = work_root / 'recovery'

if RESET_WORK_ROOT and work_root.exists():
    shutil.rmtree(work_root)
    print(f'RESET_WORK_ROOT=True: discarded {work_root} and its staged evidence.')
work_root.mkdir(parents=True, exist_ok=True)

if not RESET_WORK_ROOT and (staging_dir.exists() or recovery_dir.exists()):
    print(f'NOTE: {work_root} already holds recovery state from an earlier run:')
    if staging_dir.exists():
        print(f'  - staged evidence: {staging_dir}')
    if recovery_dir.exists():
        published = sorted(p.name for p in (recovery_dir / "bundles").glob("*")) \
            if (recovery_dir / 'bundles').is_dir() else []
        print(f'  - published bundles: {published or "none"}')
    print('  A matching run resumes from this. A mismatch aborts with a')
    print('  RecoveryBundleError -- set RESET_WORK_ROOT = True and rerun if so.')

producer_log = work_root / 'producer.log'
producer_command = [
    sys.executable,
    '-u',
    str(Path(LOCAL_PACKAGE) / 'run_schema3_observability.py'),
    '--package-root',
    LOCAL_PACKAGE,
    '--work-root',
    str(work_root),
    # Without this, a coverage-gate failure returns 1 and skips the export
    # entirely -- hours of compute produce nothing on Drive. There is no
    # resume path in resume_observability(), so rerunning this cell is a
    # full recompute, not a cheap retry. Set the flag up front: a degraded
    # run still gets its evidence published, and the gate verdict is
    # recorded in work_root/result.json for the verification cell below.
    '--allow-shortfall',
    '--export-dir',
    EXPORT_DIR,
]
print('Starting producer; live output is also saved to', producer_log)
with producer_log.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(
        producer_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
    return_code = process.wait()

# Preserve the diagnostics even if the Colab VM is restarted later. Both
# files live on VM-local scratch, and result.json carries the coverage-gate
# verdict that --allow-shortfall keeps out of the exit code.
export_path = Path(EXPORT_DIR)
for diagnostic in (producer_log, work_root / 'result.json'):
    try:
        if not diagnostic.is_file():
            continue
        export_path.mkdir(parents=True, exist_ok=True)
        shutil.copy2(diagnostic, export_path / diagnostic.name)
        print(f'Copied {diagnostic.name} to {export_path / diagnostic.name}')
    except Exception as copy_error:
        print(f'Could not copy {diagnostic.name} to Drive: {copy_error!r}')

if return_code != 0:
    raise RuntimeError(
        f'V9 producer exited with code {return_code}. --allow-shortfall is set, '
        'so this is NOT a coverage-gate failure -- the producer crashed. If it '
        'got as far as writing the artifact and only the Drive publish failed, '
        'run the re-export cell below now, while this VM is still alive, to '
        'publish from the work root without recomputing. '
        f'Full live output is in {producer_log} and the Drive copy is '
        f'{export_path / "producer.log"}.'
    )

**Manual recovery path — it is disabled by default and "Run all" will skip it.**

`_export_bundle` is the last thing the producer does, and it either publishes the whole `recovery/` tree or raises and cleans up after itself, leaving Drive untouched. If that happens, the finished artifact still exists in `/content/v9_schema3_run`, which survives until the runtime is recycled. This cell republishes from there through the same verified exporter, so a Drive hiccup at the finish line costs a retry rather than the entire run.

Use it only if the producer cell above raised: set `FORCE_REEXPORT = True` and run this cell on its own. It will fail loudly if there is no artifact to publish. Leave it `False` otherwise — on a successful run the export already happened, and repeating it just copies the whole tree to Drive again.

In [ ]:
import json
import sys
from pathlib import Path

# Manual fallback -- opt in by flipping this to True, then run this cell alone.
#
# It stays off so that "Run all" skips it: on a successful run the producer has
# already published, and re-running the exporter would copy the entire recovery
# tree to Drive a second time for nothing.
#
# Turn it on only when the producer cell raised *after* the artifact was
# written and the failure was in the Drive publish. It republishes from the
# surviving /content work root through the exact same verified exporter the
# producer uses (staged copy, per-file sha256 read-back, sidecar reference
# check, atomic directory swap), so nothing is recomputed and nothing is
# trusted that the producer would not have trusted. The work root is VM-local
# scratch: do this before the runtime is recycled or the artifact is gone.

FORCE_REEXPORT = False

if not FORCE_REEXPORT:
    print('Re-export skipped (FORCE_REEXPORT is False). This cell is a manual '
          'recovery path for a failed Drive publish.')
else:
    if LOCAL_PACKAGE not in sys.path:
        sys.path.insert(0, LOCAL_PACKAGE)
    from run_schema3_observability import _export_bundle

    WORK_ROOT = Path('/content/v9_schema3_run')
    artifact_source = WORK_ROOT / 'hybrid_recovery_explanations_v9.json'
    if not artifact_source.is_file():
        raise FileNotFoundError(
            f'No artifact to re-export at {artifact_source}. The producer never '
            'got far enough to write one, so there is nothing to publish and '
            'the run has to be repeated.'
        )

    exported = _export_bundle(artifact_source, Path(EXPORT_DIR))
    print(json.dumps(exported, indent=2, sort_keys=True))
    print('Re-export complete. Run the verification cell below.')

Final check. This confirms what actually landed on Drive — the pointer manifest, its `recovery/` evidence tree, and the canonical `bundles/<bundle_id>/manifest.json` — and then prints the coverage-gate verdict recorded in `result.json`.

Read both halves. The first says the export is complete; the second says whether the run that produced it was healthy. With `--allow-shortfall` set, those are independent questions.

In [ ]:
import json
from pathlib import Path

WORK_ROOT = Path('/content/v9_schema3_run')
export_path = Path(EXPORT_DIR)
artifact = export_path / 'hybrid_recovery_explanations_v9.json'
recovery = export_path / 'recovery'

if not artifact.is_file():
    raise FileNotFoundError(
        f'Exported artifact is missing: {artifact}. Nothing was published to '
        'Drive -- check the producer output above and try the re-export cell.'
    )
manifest = json.loads(artifact.read_text(encoding='utf-8'))
bundle_id = manifest['bundle_id']
bundle_path = manifest['bundle_path']
# _export_bundle enforces this before it publishes anything; re-checking it
# here means the sidecar paths resolved below are the canonical ones.
if bundle_path != f'bundles/{bundle_id}':
    raise ValueError(f'manifest bundle_path is not canonical: {bundle_path!r}')

for required in (recovery / 'current.json', recovery / bundle_path / 'manifest.json'):
    if not required.is_file():
        raise FileNotFoundError(f'Exported recovery tree is incomplete: {required}')

file_count = sum(1 for path in recovery.rglob('*') if path.is_file())
print(f'Artifact:      {artifact}')
print(f'Recovery tree: {recovery} ({file_count} files)')
print(f'Bundle:        {bundle_id}')
print('Export is present and structurally complete.')

# --allow-shortfall keeps a degraded run out of the exit code, so read the
# recorded verdict instead of inferring health from a cell that did not raise.
result_path = WORK_ROOT / 'result.json'
if not result_path.is_file():
    result_path = export_path / 'result.json'
if not result_path.is_file():
    print(f'\nWARNING: result.json is missing; cannot report the coverage-gate verdict.')
else:
    result = json.loads(result_path.read_text(encoding='utf-8'))
    print(f'\nGate verdict from {result_path}')
    print('coverage:', json.dumps(result.get('coverage', {}), indent=2, sort_keys=True))
    if result.get('coverage_gate_passed'):
        print('\ncoverage_gate_passed: True -- export verified complete and healthy.')
    else:
        print('\n' + '=' * 72)
        print('DEGRADED RUN. The export above is real, complete and verified,')
        print('but the coverage gate did NOT pass. Reasons:')
        for reason in result.get('coverage_gate_failed_reasons', []):
            print(f'  - {reason}')
        print('=' * 72)